# Error Analysis: Linguistic Patterns in Misclassified FPB Sentences

**Aim:** Identify systematic error patterns in ModernFinBERT's misclassifications on FinancialPhraseBank `sentences_50agree`. Categorize errors by confusion type, text length, linguistic patterns, and financial domain to guide targeted improvements.

**Model:** `neoyipeng/ModernFinBERT-base` (LoRA fine-tuned on aggregated data, FPB held out)

**Test set:** FPB `sentences_50agree` (4,846 samples)

## 1. Setup & Installation

In [ ]:
%%capture
!pip install -q "datasets>=3.4.1,<4.0.0" scikit-learn matplotlib seaborn transformers torch

In [ ]:
import json
import re
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
)
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm import tqdm

NUM_CLASSES = 3
LABEL_NAMES = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
LABEL_ID_TO_NAME = {0: "NEG", 1: "NEU", 2: "POS"}

## 2. Load Model & Data

In [ ]:
model_name = "neoyipeng/ModernFinBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()
print(f"Model loaded on {device}")
print(f"id2label: {model.config.id2label}")

In [ ]:
fpb_50 = load_dataset("financial_phrasebank", "sentences_50agree", trust_remote_code=True)["train"]
print(f"FPB 50agree: {len(fpb_50):,} samples")
print(f"Label distribution: {Counter(fpb_50['label'])}")

## 3. Run Inference & Collect Misclassifications

In [ ]:
texts = fpb_50["sentence"]
labels = fpb_50["label"]

all_preds = []
all_probs = []

with torch.no_grad():
    for i in tqdm(range(0, len(texts), 32), desc="Inference"):
        batch_texts = texts[i : i + 32]
        inputs = tokenizer(
            batch_texts, return_tensors="pt", padding=True,
            truncation=True, max_length=512,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        preds = np.argmax(logits.cpu().numpy(), axis=-1)
        all_preds.extend(preds)
        all_probs.extend(probs)

y_true = np.array(labels)
y_pred = np.array(all_preds)
y_probs = np.array(all_probs)

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
print(f"\nOverall Accuracy: {acc:.4f} ({int(acc * len(y_true))}/{len(y_true)})")
print(f"Overall Macro F1: {macro_f1:.4f}")
print(classification_report(y_true, y_pred, target_names=LABEL_NAMES))

In [ ]:
# Build DataFrame of all predictions, then filter to errors
df = pd.DataFrame({
    "text": texts,
    "true_label": y_true,
    "pred_label": y_pred,
    "true_name": [LABEL_NAMES[l] for l in y_true],
    "pred_name": [LABEL_NAMES[l] for l in y_pred],
    "confidence": [y_probs[i, y_pred[i]] for i in range(len(y_pred))],
    "word_count": [len(t.split()) for t in texts],
})
df["correct"] = df["true_label"] == df["pred_label"]
df["confusion_type"] = df.apply(
    lambda r: f"{LABEL_ID_TO_NAME[r['true_label']]}→{LABEL_ID_TO_NAME[r['pred_label']]}"
    if not r["correct"] else "correct",
    axis=1,
)

errors = df[~df["correct"]].copy()
print(f"Total samples: {len(df)}")
print(f"Correct: {df['correct'].sum()} ({df['correct'].mean():.2%})")
print(f"Errors: {len(errors)} ({len(errors)/len(df):.2%})")

## 4. Confusion Matrix Heatmap

In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax,
)
ax.set_title(f"ModernFinBERT — FPB 50agree\nAcc={acc:.2%}  Macro-F1={macro_f1:.2%}")
ax.set_ylabel("True Label")
ax.set_xlabel("Predicted Label")
plt.tight_layout()
plt.savefig("confusion_matrix_error_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved confusion_matrix_error_analysis.png")

## 5. Error Analysis by Confusion Type

Break down errors by which label pairs are confused (e.g., NEG predicted as NEU).

In [ ]:
confusion_counts = errors["confusion_type"].value_counts()
confusion_pct = (confusion_counts / len(errors) * 100).round(1)

confusion_summary = pd.DataFrame({
    "Count": confusion_counts,
    "% of Errors": confusion_pct,
    "% of Total": (confusion_counts / len(df) * 100).round(2),
})
print("Error Breakdown by Confusion Type")
print("=" * 50)
print(confusion_summary.to_string())

# Show 3-5 examples per major confusion type
print("\n")
for ctype in confusion_counts.index[:6]:
    subset = errors[errors["confusion_type"] == ctype]
    print(f"\n{'─' * 70}")
    print(f"  {ctype}  ({len(subset)} errors, {confusion_pct[ctype]:.1f}% of errors)")
    print(f"{'─' * 70}")
    for _, row in subset.head(4).iterrows():
        conf_str = f"[conf={row['confidence']:.3f}]"
        print(f"  {conf_str}  {row['text'][:120]}{'...' if len(row['text']) > 120 else ''}")
    print()

## 6. Error Analysis by Text Length

Categorize errors by sentence length: **short** (<15 words), **medium** (15-30 words), **long** (>30 words).

In [ ]:
def length_category(wc):
    if wc < 15:
        return "short (<15 words)"
    elif wc <= 30:
        return "medium (15-30 words)"
    else:
        return "long (>30 words)"

df["length_cat"] = df["word_count"].apply(length_category)
errors["length_cat"] = errors["word_count"].apply(length_category)

length_order = ["short (<15 words)", "medium (15-30 words)", "long (>30 words)"]

length_stats = []
for cat in length_order:
    total_in_cat = len(df[df["length_cat"] == cat])
    errors_in_cat = len(errors[errors["length_cat"] == cat])
    error_rate = errors_in_cat / total_in_cat if total_in_cat > 0 else 0
    length_stats.append({
        "Length Category": cat,
        "Total Samples": total_in_cat,
        "Errors": errors_in_cat,
        "Error Rate": f"{error_rate:.2%}",
        "% of All Errors": f"{errors_in_cat / len(errors) * 100:.1f}%",
    })

length_df = pd.DataFrame(length_stats)
print("Error Distribution by Text Length")
print("=" * 75)
print(length_df.to_string(index=False))

# Examples from each category
print("\n")
for cat in length_order:
    subset = errors[errors["length_cat"] == cat]
    if len(subset) == 0:
        continue
    print(f"\n{'─' * 70}")
    print(f"  {cat}  ({len(subset)} errors)")
    print(f"{'─' * 70}")
    for _, row in subset.head(3).iterrows():
        print(f"  [{row['confusion_type']}, conf={row['confidence']:.3f}] "
              f"{row['text'][:110]}{'...' if len(row['text']) > 110 else ''}")
    print()

## 7. Error Analysis by Linguistic Pattern

Tag each misclassified sentence with linguistic features that may confuse the model:
- **Hedging language**: "may", "could", "might", "possibly", "likely"
- **Conditional statements**: "if", "although", "however", "despite", "while", "unless"
- **Comparative language**: "more than", "less than", "compared to", "higher", "lower", "versus"
- **Implicit sentiment**: no explicit sentiment words — sentiment inferred from context or numbers

In [ ]:
# Linguistic pattern definitions
LINGUISTIC_PATTERNS = {
    "Hedging": r"\b(may|could|might|possibly|likely|unlikely|perhaps|potential|potentially|uncertain)\b",
    "Conditional": r"\b(if|although|however|despite|while|unless|though|nevertheless|yet|but)\b",
    "Comparative": r"\b(more than|less than|compared to|higher|lower|versus|vs\.|greater|fewer|better|worse)\b",
}

# Explicit sentiment words (used to detect implicit sentiment by their absence)
EXPLICIT_SENTIMENT = (
    r"\b(increase|decrease|rise|fall|drop|gain|loss|profit|growth|decline|"
    r"improve|deteriorat|positive|negative|strong|weak|good|bad|exceed|miss|"
    r"beat|disappoint|surge|plunge|boost|cut|slash|soar|plummet|recover)\b"
)

def tag_linguistic_patterns(text):
    """Return list of linguistic patterns present in the text."""
    text_lower = text.lower()
    tags = []
    for name, pattern in LINGUISTIC_PATTERNS.items():
        if re.search(pattern, text_lower):
            tags.append(name)
    if not re.search(EXPLICIT_SENTIMENT, text_lower):
        tags.append("Implicit sentiment")
    return tags

errors["linguistic_patterns"] = errors["text"].apply(tag_linguistic_patterns)

# Also tag all samples to compute error rate per pattern
df["linguistic_patterns"] = df["text"].apply(tag_linguistic_patterns)

# Compute stats per pattern
all_patterns = ["Hedging", "Conditional", "Comparative", "Implicit sentiment"]
pattern_stats = []

for pattern in all_patterns:
    total_with = df["linguistic_patterns"].apply(lambda ps: pattern in ps).sum()
    errors_with = errors["linguistic_patterns"].apply(lambda ps: pattern in ps).sum()
    error_rate = errors_with / total_with if total_with > 0 else 0
    baseline_rate = len(errors) / len(df)
    pattern_stats.append({
        "Linguistic Pattern": pattern,
        "Total Samples": total_with,
        "Errors": errors_with,
        "Error Rate": f"{error_rate:.2%}",
        "Baseline Error Rate": f"{baseline_rate:.2%}",
        "Relative Risk": f"{error_rate / baseline_rate:.2f}x" if baseline_rate > 0 else "N/A",
    })

pattern_df = pd.DataFrame(pattern_stats)
print("Error Distribution by Linguistic Pattern")
print("=" * 90)
print(pattern_df.to_string(index=False))
print(f"\n(Baseline error rate: {len(errors)/len(df):.2%})")
print("Relative Risk > 1.0x means this pattern is associated with higher-than-average error rate.")

In [ ]:
# Show examples per linguistic pattern
for pattern in all_patterns:
    subset = errors[errors["linguistic_patterns"].apply(lambda ps: pattern in ps)]
    if len(subset) == 0:
        continue
    print(f"\n{'─' * 70}")
    print(f"  {pattern}  ({len(subset)} errors)")
    print(f"{'─' * 70}")
    for _, row in subset.head(5).iterrows():
        print(f"  [{row['confusion_type']}, conf={row['confidence']:.3f}]")
        print(f"    {row['text'][:130]}{'...' if len(row['text']) > 130 else ''}")
    print()

## 8. Error Analysis by Financial Domain

Categorize sentences into financial domains based on keyword matching:
- **Mining / Resources**: mining, ore, drill, hectare, gold, copper, exploration
- **Earnings**: profit, loss, revenue, earnings, EPS, net income, operating result
- **Market / Trading**: shares, stock, trading, market, dividend, investor, index
- **General Business**: company, CEO, agreement, acquisition, partnership, contract

In [ ]:
DOMAIN_PATTERNS = {
    "Mining / Resources": r"\b(mining|mine|ore|drill|hectare|gold|copper|zinc|nickel|exploration|mineral|deposit|assay)\b",
    "Earnings": r"\b(profit|loss|revenue|earnings|EPS|net income|operating result|EBIT|EBITDA|turnover|sales)\b",
    "Market / Trading": r"\b(shares|stock|trading|market|dividend|investor|index|IPO|listing|NYSE|HEL|OMX)\b",
    "General Business": r"\b(company|CEO|agreement|acquisition|partnership|contract|merger|joint venture|subsidiary|restructur)\b",
}

def classify_domain(text):
    """Assign the first matching domain, or 'Other' if none match."""
    text_lower = text.lower()
    for domain, pattern in DOMAIN_PATTERNS.items():
        if re.search(pattern, text_lower, re.IGNORECASE):
            return domain
    return "Other"

df["domain"] = df["text"].apply(classify_domain)
errors["domain"] = errors["text"].apply(classify_domain)

domain_stats = []
for domain in list(DOMAIN_PATTERNS.keys()) + ["Other"]:
    total_in = len(df[df["domain"] == domain])
    errors_in = len(errors[errors["domain"] == domain])
    error_rate = errors_in / total_in if total_in > 0 else 0
    domain_stats.append({
        "Domain": domain,
        "Total Samples": total_in,
        "Errors": errors_in,
        "Error Rate": f"{error_rate:.2%}",
        "% of All Errors": f"{errors_in / len(errors) * 100:.1f}%",
    })

domain_df = pd.DataFrame(domain_stats)
print("Error Distribution by Financial Domain")
print("=" * 75)
print(domain_df.to_string(index=False))

# Examples
print("\n")
for domain in list(DOMAIN_PATTERNS.keys()) + ["Other"]:
    subset = errors[errors["domain"] == domain]
    if len(subset) == 0:
        continue
    print(f"\n{'─' * 70}")
    print(f"  {domain}  ({len(subset)} errors)")
    print(f"{'─' * 70}")
    for _, row in subset.head(3).iterrows():
        print(f"  [{row['confusion_type']}, conf={row['confidence']:.3f}]")
        print(f"    {row['text'][:130]}{'...' if len(row['text']) > 130 else ''}")
    print()

## 9. Summary Table & Visualizations

In [ ]:
# Visualization: error rate by category
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Error rate by confusion type
ax = axes[0]
ct_data = confusion_counts.head(6)
colors = sns.color_palette("Reds_r", len(ct_data))
bars = ax.barh(ct_data.index[::-1], ct_data.values[::-1], color=colors[::-1], edgecolor="white")
ax.set_xlabel("Count")
ax.set_title("Errors by Confusion Type")
for bar, val in zip(bars, ct_data.values[::-1]):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            str(val), va="center", fontsize=10)

# 2. Error rate by text length
ax = axes[1]
len_rates = []
for cat in length_order:
    total_in_cat = len(df[df["length_cat"] == cat])
    errors_in_cat = len(errors[errors["length_cat"] == cat])
    len_rates.append(errors_in_cat / total_in_cat if total_in_cat > 0 else 0)
bars = ax.bar(range(len(length_order)), len_rates, color=["#42A5F5", "#66BB6A", "#FFA726"], edgecolor="white")
ax.set_xticks(range(len(length_order)))
ax.set_xticklabels(["Short\n(<15w)", "Medium\n(15-30w)", "Long\n(>30w)"])
ax.set_ylabel("Error Rate")
ax.set_title("Error Rate by Text Length")
ax.axhline(y=len(errors) / len(df), color="red", linestyle="--", alpha=0.7, label="Baseline")
ax.legend()
for bar, rate in zip(bars, len_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{rate:.1%}", ha="center", fontsize=10)

# 3. Error rate by linguistic pattern
ax = axes[2]
pat_rates = []
for pattern in all_patterns:
    total_with = df["linguistic_patterns"].apply(lambda ps: pattern in ps).sum()
    errors_with = errors["linguistic_patterns"].apply(lambda ps: pattern in ps).sum()
    pat_rates.append(errors_with / total_with if total_with > 0 else 0)
bars = ax.bar(range(len(all_patterns)), pat_rates, color=sns.color_palette("Set2", len(all_patterns)), edgecolor="white")
ax.set_xticks(range(len(all_patterns)))
ax.set_xticklabels(["Hedging", "Conditional", "Comparative", "Implicit\nSentiment"], fontsize=9)
ax.set_ylabel("Error Rate")
ax.set_title("Error Rate by Linguistic Pattern")
ax.axhline(y=len(errors) / len(df), color="red", linestyle="--", alpha=0.7, label="Baseline")
ax.legend()
for bar, rate in zip(bars, pat_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{rate:.1%}", ha="center", fontsize=10)

plt.suptitle("ModernFinBERT Error Analysis — FPB 50agree", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("error_analysis_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved error_analysis_summary.png")

## 10. Confidence Analysis on Errors

Are misclassified samples associated with lower model confidence, or does the model fail confidently?

In [ ]:
correct_conf = df[df["correct"]]["confidence"]
error_conf = errors["confidence"]

print(f"Correct predictions — mean confidence: {correct_conf.mean():.4f}, median: {correct_conf.median():.4f}")
print(f"Misclassified      — mean confidence: {error_conf.mean():.4f}, median: {error_conf.median():.4f}")
print(f"\nHigh-confidence errors (conf > 0.9): {(error_conf > 0.9).sum()} / {len(errors)} "
      f"({(error_conf > 0.9).mean():.1%})")
print(f"Low-confidence errors (conf < 0.5):  {(error_conf < 0.5).sum()} / {len(errors)} "
      f"({(error_conf < 0.5).mean():.1%})")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(correct_conf, bins=30, alpha=0.6, label="Correct", color="#66BB6A", density=True)
ax.hist(error_conf, bins=30, alpha=0.6, label="Misclassified", color="#EF5350", density=True)
ax.set_xlabel("Prediction Confidence")
ax.set_ylabel("Density")
ax.set_title("Confidence Distribution: Correct vs Misclassified")
ax.legend()
plt.tight_layout()
plt.show()

## 11. Save Results

In [ ]:
import os

results_dir = "../results"
os.makedirs(results_dir, exist_ok=True)

# Build comprehensive results dict
error_analysis_results = {
    "model": model_name,
    "test_set": "FPB sentences_50agree",
    "total_samples": len(df),
    "total_errors": len(errors),
    "accuracy": round(acc, 4),
    "macro_f1": round(macro_f1, 4),
    "confusion_type": {
        ctype: {
            "count": int(count),
            "pct_of_errors": round(count / len(errors) * 100, 1),
        }
        for ctype, count in confusion_counts.items()
    },
    "text_length": {
        row["Length Category"]: {
            "total_samples": row["Total Samples"],
            "errors": row["Errors"],
            "error_rate": row["Error Rate"],
        }
        for _, row in length_df.iterrows()
    },
    "linguistic_patterns": {
        row["Linguistic Pattern"]: {
            "total_samples": row["Total Samples"],
            "errors": row["Errors"],
            "error_rate": row["Error Rate"],
            "relative_risk": row["Relative Risk"],
        }
        for _, row in pattern_df.iterrows()
    },
    "domain": {
        row["Domain"]: {
            "total_samples": row["Total Samples"],
            "errors": row["Errors"],
            "error_rate": row["Error Rate"],
        }
        for _, row in domain_df.iterrows()
    },
    "confidence": {
        "correct_mean": round(float(correct_conf.mean()), 4),
        "correct_median": round(float(correct_conf.median()), 4),
        "error_mean": round(float(error_conf.mean()), 4),
        "error_median": round(float(error_conf.median()), 4),
        "high_conf_errors_pct": round(float((error_conf > 0.9).mean()) * 100, 1),
        "low_conf_errors_pct": round(float((error_conf < 0.5).mean()) * 100, 1),
    },
    "error_examples": {
        ctype: [
            {"text": row["text"], "confidence": round(row["confidence"], 4)}
            for _, row in errors[errors["confusion_type"] == ctype].head(3).iterrows()
        ]
        for ctype in confusion_counts.index[:6]
    },
}

results_path = os.path.join(results_dir, "error_analysis.json")
with open(results_path, "w") as f:
    json.dump(error_analysis_results, f, indent=2, default=str)

print(f"Results saved to {results_path}")
print(f"Keys: {list(error_analysis_results.keys())}")